# 🎒 Fine-Tuning paraphrase-multilingual-MiniLM-L12-v2 for Product Matching

This notebook trains/fine-tunes the SentenceTransformer model to map school supply lists to product catalog entries.
It addresses: 
- Bigrams representing quantity + article
- Generic 'DinA' mapping to both A4 and A5 products
- Noun-color associations and word proximity rules

In [ ]:
import pandas as pd
import os
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

## 1. Load Data

In [ ]:
df_train = pd.read_csv("data/training_set.csv")
df_products = pd.read_csv("data/products.csv")
products_dict = df_products.set_index('id').to_dict('index')

print(f"Loaded {len(df_train)} training records and {len(df_products)} catalog products.")

## 2. Prepare Training Examples and Augmentations

In [ ]:
train_examples = []

# 1. Add mapped examples from PDFs
for _, row in df_train.iterrows():
    prod_id = row['product']
    if prod_id != 0 and prod_id in products_dict:
        prod = products_dict[prod_id]
        anchor = str(row['raw_line'])
        positive = f"{prod['name']} {prod['brand']} {prod.get('description', '')}".strip()
        train_examples.append(InputExample(texts=[anchor, positive]))

# 2. Synthetic augmentation to improve specific cases (Bigrams, DinA, Colors)
synthetic_pairs = []

# A. Bigrams (Quantity + Article)
quantities = ["1", "2", "5", "10", "3x", "4 Stk."]
articles = {
    "heft": [1001, 1002, 1034, 1035],
    "schnellhefter": [1015, 1016, 1017, 1046],
    "umschlag": [1021, 1022, 1023, 1024],
    "bleistift": [1004],
    "buntstifte": [1005]
}

for qty in quantities:
    for art, ids in articles.items():
        for pid in ids:
            prod = products_dict[pid]
            anchor = f"{qty} {art}"
            positive = f"{prod['name']} {prod['brand']} {prod.get('description', '')}".strip()
            synthetic_pairs.append(InputExample(texts=[anchor, positive]))

# B. Generic 'DinA' mapped to both A4 and A5
for pid in [1001, 1002, 1021, 1022, 1023, 1031, 1034, 1035, 1038]:
    prod = products_dict[pid]
    anchor_dina = f"dina {prod['name'].split(',')[0]}"
    positive = f"{prod['name']} {prod['brand']} {prod.get('description', '')}".strip()
    synthetic_pairs.append(InputExample(texts=[anchor_dina, positive]))

# C. Color proximity cases
# e.g. 'Schnellhefter rot' -> matched specifically to red Schnellhefter
colors = ["rot", "blau", "grün", "gelb", "weiß", "lila", "schwarz"]
for color in colors:
    # Match folders
    folders = df_products[df_products['name'].str.lower().str.contains("schnellhefter") & df_products['name'].str.lower().str.contains(color)]
    for _, prod in folders.iterrows():
        anchor = f"schnellhefter {color}"
        positive = f"{prod['name']} {prod['brand']} {prod.get('description', '')}".strip()
        synthetic_pairs.append(InputExample(texts=[anchor, positive]))
        
    # Match covers
    covers = df_products[df_products['name'].str.lower().str.contains("umschlag") & df_products['name'].str.lower().str.contains(color)]
    for _, prod in covers.iterrows():
        anchor = f"umschlag {color}"
        positive = f"{prod['name']} {prod['brand']} {prod.get('description', '')}".strip()
        synthetic_pairs.append(InputExample(texts=[anchor, positive]))

train_examples.extend(synthetic_pairs)
print(f"Total training examples: {len(train_examples)} (including {len(synthetic_pairs)} synthetic additions).")

## 3. Fine-Tune SentenceTransformer Model

In [ ]:
model_name = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Fine-tune model for 4 epochs
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=100,
    show_progress_bar=True
)

## 4. Save Fine-Tuned Model

In [ ]:
save_path = "src/fine_tuned_sentence_transformer"
model.save(save_path)
print(f"Model successfully saved to {save_path}")

## 5. Evaluate and Verify

In [ ]:
from sentence_transformers import util

test_queries = [
    "5 Hefte",
    "DinA Umschlag rot",
    "DinA4 Heft liniert Lineatur 21",
    "Schnellhefter blau"
]

for query in test_queries:
    query_emb = model.encode(query, convert_to_tensor=True)
    best_score = -1
    best_prod = None
    for _, prod in df_products.iterrows():
        prod_text = f"{prod['name']} {prod['brand']} {prod.get('description', '')}".strip()
        prod_emb = model.encode(prod_text, convert_to_tensor=True)
        score = float(util.cos_sim(query_emb, prod_emb)[0][0])
        if score > best_score:
            best_score = score
            best_prod = prod
    print(f"Query: '{query}' -> Best match: {best_prod['name']} (Score: {best_score:.4f})")